[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day4_practice.ipynb)

# Day 4 · 수강생끼리 — 딥러닝

클래스 · 텐서 · 학습 루프 · 평가

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

강의가 끝난 뒤 **수강생끼리** 푼다.

`스스로 풀기` 는 각자, `조별 과제` 는 2~3명이 한 조로 상의하며 푼다.
막히면 강의 노트북(`lecture`)의 실습 셀로 돌아가 확인한다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 클래스

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 2.** `Kiln` 에 정격 온도를 넘었는지 판정하는 `over(temp)` 를 넣는다.

In [ ]:
class Machine:
    def __init__(self, name):
        self.name = name

class Kiln(Machine):
    def __init__(self, name, rated):
        super().__init__(name)
        self.rated = rated

    def over(self, temp):
        return ___

k = Kiln('C', 870)

assert k.over(898) is True and k.over(850) is False, '판정이 틀렸다'
print('통과')

## 2. 텐서

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 4.** `w = 2.0` 일 때 `loss = (w - 7) ** 2` 의 기울기를 구해 `g` 에 담는다.

In [ ]:
import torch
w = torch.tensor(2.0, requires_grad=True)
loss = ___
loss.backward()
g = w.grad.item()

assert abs(g - (-10.0)) < 1e-6, f'기대 -10.0, 실제 {g}'
print('통과')

## 3. 모델과 학습 루프

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 7.** `opt.zero_grad()` 를 **빼면** 어떻게 되는지 본다.
기울기가 쌓여 손실이 제대로 안 줄어드는 것을 확인한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.SGD(model.parameters(), lr=0.5)
grads = []
for epoch in range(20):
    loss = lossfn(model(X_tr), y_tr)
    ___   # zero_grad 를 빼 본다
    loss.backward()
    opt.step()
    grads.append(model.fc1.weight.grad.abs().mean().item())

assert grads[-1] > grads[0], f'기울기가 쌓여 커진다: {grads[0]:.4f} → {grads[-1]:.4f}'
print(f'통과 — 기울기가 {grads[0]:.4f} 에서 {grads[-1]:.4f} 로 쌓였다')

### 조별 과제

2~3명이 한 조로 상의하며 푼다.

> **실습문제 1.** 학습하면서 **손실 곡선**을 그린다.
200회 돌리며 매 회 손실을 모아 `losses` 에 담고 그린다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
import matplotlib.pyplot as plt
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
# 여기에 작성한다

assert len(losses) == 200, f'200개여야 한다: {len(losses)}'
assert losses[-1] < losses[0] / 2, f'절반 아래로 떨어져야 한다: {losses[0]:.3f} → {losses[-1]:.3f}'
print('통과')

## 4. 평가

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 8.** 학습한 모델의 **테스트 정확도**를 `acc` 에 담는다.
> `torch.no_grad()` 안에서 계산한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

X_tr, X_te, y_tr, y_te = tensors()
model = MLP(X_tr.shape[1])
lossfn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for _ in range(300):
    loss = lossfn(model(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    pred = ___
    acc = ___

assert acc > 0.85, f'0.85 는 넘어야 한다: {acc}'
print('통과 — 정확도', round(acc, 3))

### 조별 과제

2~3명이 한 조로 상의하며 푼다.

> **실습문제 2.** **shape 에러를 일부러 내 보고** 메시지를 읽는다.
입력 열 수와 `nn.Linear` 의 첫 인자를 다르게 주면 무슨 말이 나오는지 확인한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
X_tr, X_te, y_tr, y_te = tensors()
# 여기에 작성한다

assert 'mat1' in msg or 'shape' in msg.lower(), f'shape 관련 메시지여야 한다: {msg}'
print('통과')

## 5. 종합 문제

### 조별 과제

2~3명이 한 조로 상의하며 푼다.

> **실습문제 3.** **은닉층 크기를 바꿔 가며** 정확도를 비교한다.
8 · 16 · 32 · 64 로 각각 300회 학습해 `results` 딕셔너리에 담고 출력한다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
X_tr, X_te, y_tr, y_te = tensors()
# 여기에 작성한다

assert len(results) == 4, f'4가지여야 한다: {results}'
assert all(v > 0.8 for v in results.values()), f'전부 0.8 은 넘는다: {results}'
print('통과')

> **실습문제 4.** **회귀로 바꿔 본다.** `방전용량` 을 맞히는 신경망을 만든다.
> 마지막 층은 그대로 1개, 손실은 `nn.MSELoss()` 를 쓴다.
> 정답 스케일이 크므로 `y` 도 표준화하면 학습이 훨씬 잘 된다.

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

def tensors(target='양품여부'):
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    d = pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)
    X = d.drop(columns=['양품여부', '방전용량', '배치번호']).astype('float32')
    y = d[target].astype('float32')
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42,
        stratify=y if target == '양품여부' else None)
    sc = StandardScaler()
    to = lambda a: torch.tensor(np.asarray(a, dtype='float32'))
    return (to(sc.fit_transform(X_tr)), to(sc.transform(X_te)),
            to(y_tr.values).unsqueeze(1), to(y_te.values).unsqueeze(1))
X_tr, X_te, y_tr, y_te = tensors(target='방전용량')
# 여기에 작성한다

assert rmse < 4.0, f'RMSE 4 미만은 나온다: {rmse}'
print('통과')